* **Original paper:** [*Tessa C, Lucetti C, Giannelli M, Diciotti S, Poletti M, Danti S, Baldacci F, Vignali C, Bonuccelli U, Mascalchi M, Toschi N. Progression of brain atrophy in the early stages of Parkinson's disease: a longitudinal tensor-based morphometry study in de novo patients without cognitive impairment. Hum Brain Mapp. 2014 Aug;35(8):3932-44. doi: 10.1002/hbm.22449. Epub 2014 Jan 22. PMID: 24453162; PMCID: PMC6868950.*](https://onlinelibrary.wiley.com/doi/abs/10.1002/hbm.22449)

# Introduction

* **Cohort quote from the paper:** "*Overall, 22 patients (4 women and 18 men, mean age 61.5 ± 8.8) and 17 control subjects (8 women and 9 men, mean age 59.1 ± 8.5 years) completed the study and underwent a second MRI examination. The mean (± standard deviation) follow-up time for patients and controls was 2.8 ± 0.6 (range 2–4) years and 3.9 ± 2.2 (range 2–7) years, respectively. Differences for age between PD patients and control subjects were not significant (P = 0.48, MannWhitney U-test)*"


* Demographics for the PD patients (table taken from the paper):
<img src="./images/original-cohort.png" alt= “” width="30%" height="30%">


* At the baseline evaluation "*No difference in local volume between patients and control subjects was revealed*".


* A (very) brief summary of the main results of the logtitudinal evaluation is:


  * Control subjects: Baseline versus follow-up
    * Control subjects experienced atrophy in several white matter and grey matter regions (Fig. 1a), and cerebrospinal fluid enlargement. There were atrophy clusters involved mainly in white matter, and were more widespread in the frontal lobe.
    
    
  * PD patients: Baseline versus follow-up
    * PD patients showed clusters of reduced white and grey matter volume. These were more evident in the white matter, specially the frontal lobe (Fig. 1b), and showed cerebrospinal fluid enlargement. Grey matter involvement was more widespread than in the control subjects.
    <img src="./images/original-fig1.png" alt= “” width="50%" height="50%">
    
  * PD patients versus control subjects
    * "*PD patients developed bilateral clusters of increased atrophy*" (Fig. 2).
    <img src="./images/original-fig2.png" alt= “” width="50%" height="50%">
    
    
  * Correlation analyses
    * "*In PD patients, no significant correlation between warprates and motor or neuropsychological test scores or their average changes per year between baseline and follow-up were identified*".

# Setup

In [1]:
import numpy as np
import pandas as pd
import os

import livingpark_utils
from livingpark_utils import download

In [2]:
inputs_dir = os.path.join(os.getcwd(), "inputs/study_files")
outputs_dir = os.path.join(os.getcwd(), "outputs")
data_dir = os.path.join(os.getcwd(), "data")

utils = livingpark_utils.LivingParkUtils()
downloader = download.ppmi.Downloader(utils.study_files_dir)
#random_seed = 1
utils.notebook_init()

This notebook was run on 2023-03-10 07:18:13 UTC +0000


# PPMI cohort preparation

* ### PPMI study data files to reconstruct cohort:
  * T1w images (baseline and 24 month follow-up)
  * Age at visit (Age)
  * Demographics (Sex)
  * Participant Status (Parkinson's vs healthy diagnosis - *note the original uses UK Brain Bank criteria, and a neurological examination and levodopa challenge test for idiopathic diagnosis*)
  * Unified Parkinson's Disease Rating Scale (UPDRS) Parts I, II and III (Parkinson severity - *note that III includes the Hoehn–Yahr (HY) staging system seen in the paper*)
  * Geriatric Depression Scale Short Version (Depression screening - exclusion criteria)
  * Montreal Cognitive Assessment MoCA (global cognitive status - MCI exclusion criteria - *note that it contains the Mini Mental State Examination seen in the paper*)
  * Cognitive Categorization (PD-MCI COGSTATE for exclusion criteria - instead of the paper's battery of standardized neuropsychological tests)
  * PD Diagnosis History (disease duration)
  * Primary Clinical Diagnosis (Controls - personal neurological diseases and normal neurological examination)

  
* ### Some stuff to consider
  * While it's not an inclusion/exclusion criterion, the paper states that  "At follow-up examination, all patients were receiving L-dopa". It makes no mention of L-dopa at baseline.
  * UPDRS, HY, MMSE are not inclusion/exclusion criteria, but where possible will be chosen to be similar to Table. 1 at baseline and follow-up.
  * For the control subjects, the original study states that they must have "No history of familial or personal neurological diseases". I can't find this detailed familial data in PPMI, the existing familial data is only with regard to PD.
  
* ### Inclusion/Exclusion criteria I will use to replicate this study:
  * Baseline and ~24 month follow-up T1-weighted MRIs available and usable for TBM.
  * Disease duration ~1 year at baseline
  * No MCI or dementia.
  * No depression.
  * No cardio-vascular autonomic dysfunction.
  * Subjects: 4 women and 18 men, Controls: 8 women and 9 men.
  * Controls: a normal neurological examination, and no relatives with PD (*See the notes above on how this differs from the original study*).  

In [3]:
required_files = [
    "Demographics.csv",
    "Age_at_visit.csv",
    "Montreal_Cognitive_Assessment__MoCA_.csv",
    "PD_Diagnosis_History.csv",
    "Cognitive_Categorization.csv",
    "Participant_Status.csv",
    "Primary_Clinical_Diagnosis.csv",
    "Geriatric_Depression_Scale__Short_Version_.csv",
    "Family_History.csv",
    "General_Physical_Exam.csv",
    "Neurological_Exam.csv",
    "Inclusion_Exclusion.csv",
    "Magnetic_Resonance_Imaging__MRI_.csv",
    #"MDS-UPDRS_Part_I.csv",
    #"MDS_UPDRS_Part_II__Patient_Questionnaire.csv",
    "MDS-UPDRS_Part_IV__Motor_Complications.csv",
    "MDS-UPDRS_Part_III.csv",
    #"MDS-UPDRS_Part_I_Patient_Questionnaire.csv",
    "Medical_Conditions_Log.csv"
]

utils.get_study_files(required_files, default=downloader)

Download skipped: No missing files!


In [4]:
#age
age_df = pd.read_csv(os.path.join(inputs_dir, "Age_at_visit.csv"), usecols=["PATNO", "EVENT_ID", "AGE_AT_VISIT"])

#sex
sex_df = pd.read_csv(os.path.join(inputs_dir,"Demographics.csv"), usecols=["PATNO", "SEX"])

#disease duration
disease_duration_df = pd.read_csv(os.path.join(inputs_dir,"PD_Diagnosis_History.csv"), usecols=["PATNO", "EVENT_ID", "PDDXDT"])

#moca and mmse
moca_df = pd.read_csv(os.path.join(inputs_dir,"Montreal_Cognitive_Assessment__MoCA_.csv"), usecols=["PATNO", "EVENT_ID", "MCATOT"])

#gdss
gdss_df = pd.read_csv(os.path.join(inputs_dir,"Geriatric_Depression_Scale__Short_Version_.csv"))

#updrs - currently just HY and III, get other stuff too
updrs3_df = pd.read_csv(os.path.join(inputs_dir,"MDS-UPDRS_Part_III.csv"), usecols=["PATNO", "EVENT_ID", "PDSTATE", "PDTRTMNT", "NP3TOT", "NHY"])

#diagnosis
diagnosis_df = pd.read_csv(os.path.join(inputs_dir, "Primary_Clinical_Diagnosis.csv"), usecols=["PATNO", "EVENT_ID", "PRIMDIAG", "OTHNEURO"])

# Cognitive Categorization
cog_cat_df = pd.read_csv(os.path.join(inputs_dir, "Cognitive_Categorization.csv"), usecols=["PATNO", "EVENT_ID", "COGSTATE"])

#mri availability
mri_df = pd.read_csv(os.path.join(inputs_dir,"Magnetic_Resonance_Imaging__MRI_.csv"))

#medical condition
med_cond_df = pd.read_csv(os.path.join(inputs_dir, "Medical_Conditions_Log.csv"), usecols=["PATNO", "EVENT_ID", "MHCAT"]).groupby(['PATNO', 'EVENT_ID'])[['MHCAT']].aggregate(lambda x: tuple(set(x))) # aggregate all codes in a tuple

#parkinsons vs healthy
part_stat_fd = pd.read_csv(os.path.join(inputs_dir, "Participant_Status.csv"))

#family history
part_stat_fd = pd.read_csv(os.path.join(inputs_dir, "Family_History.csv"))

#physical
physical_df = pd.read_csv(os.path.join(inputs_dir, "General_Physical_Exam.csv"))

#neuro
neuro_df = pd.read_csv(os.path.join(inputs_dir, "Neurological_Exam.csv"))

#inc/exc
incexc_df = pd.read_csv(os.path.join(inputs_dir, "Inclusion_Exclusion.csv"))

# PPMI cohort matching (data aggregation)